# One shape, four discretizations

The same normalization $Q = \Lambda K M \Lambda$ applies to any representation of
a shape, as it only requires a kernel and masses.

Some default choices can be found here:

| modality | kernel | masses |
|---|---|---|
| point cloud | dense/k-NN Gaussian | uniform (or KDE) |
| voxel grid | separable convolution (matrix-free) | KDE truncation correction |
| Gaussian mixture | anisotropic $C_{ij} = \sigma^2 I + \Sigma_i + \Sigma_j$ | mixture weights |
| mesh (reference) | — | FEM mass matrix |

This notebook needs the `[examples]` extra (`pyvista`, `tetgen`, `scikit-learn`):

```bash
pip install -e ".[examples]"
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

from sinkhornkernels import (
    NormalizedKernel,
    gaussian_diffusion,
    diffusion_eigsh,
    laplacian_eigenvalues,
    gmm_diffusion,
    gmm_effective_sigma2,
    sinkhorn,
)
from sinkhornkernels import mesh as skmesh
from sinkhornkernels.grid import (
    GridGaussian,
    boundary_mask,
    grid_coordinates,
    kde_masses,
    voxelize_mesh,
)

import plot_utils as plu

# pick the 3D rendering backend once: "pyvista" (interactive) or "matplotlib"
plu.set_backend("pyvista")

rng = np.random.default_rng(0)
SIGMA = 0.05  # world-units bandwidth shared by all modalities
K_EIG = 40

## Reference: FEM Laplacian on the triangle mesh

In [ ]:
vertices, faces = skmesh.load_obj("../data/armadillo.obj")
vertices -= vertices.mean(axis=0)
vertices /= np.linalg.norm(vertices, axis=1).max()

L = skmesh.cotangent_laplacian(vertices, faces)
M = skmesh.fem_mass_matrix(vertices, faces)
evals_fem, evecs_fem = skmesh.fem_spectrum(L, M, k=K_EIG)
evals_fem[:6]

## Point cloud

In [ ]:
N_pc = 4000
points = skmesh.sample_surface(vertices, faces, N_pc, rng=rng)
masses_pc = np.full(N_pc, 1.0 / N_pc)

Q_pc = gaussian_diffusion(points, SIGMA, masses=masses_pc, n_iter=20)
evalsQ_pc, evecs_pc = diffusion_eigsh(Q_pc, k=K_EIG)
evals_pc = laplacian_eigenvalues(evalsQ_pc, SIGMA)
print(f"marginal error: {Q_pc.marginal_error():.2e}")

## Voxel grid (matrix-free convolution)

The kernel is never materialized: each application is a separable Gaussian
convolution restricted to the occupied voxels. We use the one-voxel-thick
boundary shell as a surface discretization, with the KDE masses:

In [ ]:
mask_solid, origin, spacing = voxelize_mesh(vertices, faces, density=SIGMA)
mask_shell = boundary_mask(mask_solid)
print(f"grid {mask_solid.shape}, {mask_solid.sum()} solid / {mask_shell.sum()} shell voxels")

sigma_px = 1.0  # = SIGMA in world units, since spacing == SIGMA
K_vox = GridGaussian(mask_shell, sigma=sigma_px)
masses_vox = kde_masses(mask_shell, sigma=sigma_px)

Q_vox = NormalizedKernel(K_vox, masses=masses_vox, mode="sinkhorn", n_iter=20)
evalsQ_vox, evecs_vox = diffusion_eigsh(Q_vox, k=K_EIG)
evals_vox = laplacian_eigenvalues(evalsQ_vox, sigma_px * spacing)
print(f"marginal error: {Q_vox.marginal_error():.2e}")

## Gaussian mixture

A 300-component GMM fitted to surface samples. The kernel is the anisotropic
$K_{ij} = \exp(-\tfrac12 x_i^\top (\sigma^2 I + \Sigma_i + \Sigma_j)^{-1} x_j)$,
the masses are the mixture weights, and the effective bandwidth used for the
eigenvalue conversion accounts for the average blob size. `gmm_diffusion`
normalizes in the log domain, so it stays stable for all values of $\sigma$:

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=300, covariance_type="full", random_state=0)
gmm.fit(skmesh.sample_surface(vertices, faces, 10000, rng=rng))

Q_gmm = gmm_diffusion(gmm.means_, gmm.covariances_, SIGMA, masses=gmm.weights_, n_iter=100)
evalsQ_gmm, evecs_gmm = diffusion_eigsh(Q_gmm, k=K_EIG)

sigma2_eff = gmm_effective_sigma2(SIGMA, gmm.covariances_, dim=2, masses=gmm.weights_)
evals_gmm = laplacian_eigenvalues(evalsQ_gmm, np.sqrt(sigma2_eff))
print(f"marginal error: {Q_gmm.marginal_error():.2e}, effective sigma: {np.sqrt(sigma2_eff):.3f}")

## Spectra across modalities

All spectra follow the FEM reference **qualitatively** — the
$-\tfrac{2}{\sigma^2}\log\lambda^Q$ conversion is a heuristic, so we compare
shapes of the curves, not values:

In [ ]:
plt.figure(figsize=(6, 4.2))
plt.plot(evals_fem, "k.-", label="FEM cotangent (mesh)")
plt.plot(evals_pc, ".-", label="point cloud")
plt.plot(evals_vox, ".-", label="voxel shell")
plt.plot(evals_gmm, ".-", label="Gaussian mixture")
plt.xlabel("eigenvalue index")
plt.ylabel("Laplacian eigenvalue estimate")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Spectra across discretizations (qualitative)")
plt.tight_layout()

## The same eigenfunction everywhere

Low-frequency eigenvectors are consistent across representations (up to sign
and eigenspace rotations for near-degenerate eigenvalues):

In [ ]:
IDX = 3
vox_pts = grid_coordinates(mask_shell, origin=origin, spacing=spacing)


def aligned(pts_ref, vec_ref, pts, vec):
    # align the sign against the reference eigenvector via nearest neighbors
    from scipy.spatial import cKDTree

    _, nn = cKDTree(pts_ref).query(pts)
    return vec * np.sign(np.sum(vec_ref[nn] * vec) or 1.0)


panels = [
    plu.Mesh(vertices, faces, evecs_fem[:, IDX], title="mesh (FEM)"),
    plu.Points(
        points,
        aligned(vertices, evecs_fem[:, IDX], points, evecs_pc[:, IDX]),
        size=2,
        title="point cloud",
    ),
    plu.Voxels(
        vox_pts,
        spacing,
        aligned(vertices, evecs_fem[:, IDX], vox_pts, evecs_vox[:, IDX]),
        size=6,
        title="voxel shell",
    ),
    plu.Gaussians(
        gmm.means_,
        gmm.covariances_,
        aligned(vertices, evecs_fem[:, IDX], gmm.means_, evecs_gmm[:, IDX]),
        weights=gmm.weights_,
        size=25,
        title="GMM",
    ),
]
plu.plot_panels(panels, cmap="coolwarm")

## Volume version (tet mesh reference)

The same pipeline runs on volumetric discretizations: FEM reference from a
tetrahedral mesh, kernel operators on interior samples / solid voxels
(`dim=3` in the GMM conversion). Requires `tetgen`:

In [ ]:
try:
    import tetgen

    tet = tetgen.TetGen(pv.make_tri_mesh(vertices, faces))
    tet_verts, tets = tet.tetrahedralize(order=1, mindihedral=10, minratio=1.5)
    L3 = skmesh.tet_laplacian(tet_verts, tets)
    M3 = skmesh.fem_mass_matrix_tet(tet_verts, tets)
    evals_fem3, _ = skmesh.fem_spectrum(L3, M3, k=K_EIG)

    Q_vol = NormalizedKernel(
        GridGaussian(mask_solid, sigma=sigma_px),
        masses=np.ones(int(mask_solid.sum())),
        mode="sinkhorn",
        n_iter=20,
    )
    evalsQ_vol, _ = diffusion_eigsh(Q_vol, k=K_EIG)
    evals_vol = laplacian_eigenvalues(evalsQ_vol, sigma_px * spacing)

    plt.figure(figsize=(6, 4))
    plt.plot(evals_fem3, "k.-", label="FEM tetrahedral (volume mesh)")
    plt.plot(evals_vol, ".-", label="solid voxels")
    plt.xlabel("eigenvalue index")
    plt.ylabel("Laplacian eigenvalue estimate")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.title("Volume spectra (qualitative)")
    plt.tight_layout()
except ImportError:
    print("tetgen not installed - skipping the volume comparison")